# 1. What you'll learn

        - compare PCA, UMAP, and t-SNE on identical digits
- measure local neighbourhood trustworthiness
- demonstrate unstable between-cluster geometry

        **The one question this notebook answers:** What can a nonlinear embedding reveal locally, and why should distances between plotted clusters not be treated as measurements?

# 2. Setup

This lesson keeps umap and t-sne versus pca small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE,trustworthiness
from sklearn.model_selection import train_test_split
import umap

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

A stratified 1,200-row digits subset keeps CPU runtime low. Pixel vectors are scaled once from the fitting subset; labels only colour plots and evaluate neighbourhoods.

**Small example:** If five points have neighbours {B,C}, {A,C}, {A,B}, {E}, {D}, an embedding can preserve those pairs while placing the ABC island arbitrarily far from DE; local agreement does not fix global distance.

In [ ]:
d=load_digits();ids,_=train_test_split(np.arange(len(d.data)),train_size=1200,stratify=d.target,random_state=SEED);X=d.data[ids];y=d.target[ids];print("Subset:",X.shape,"classes:",np.bincount(y).tolist())

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].imshow(np.hstack([X[i].reshape(8,8) for i in range(10)]),cmap="gray_r");axes[0].set_title("Ten examples");axes[0].axis("off")
axes[1].hist(np.linalg.norm(X[:,None,:]-X[np.arange(len(X))[:1],None,:],axis=2).ravel()[:1200],bins=25,color="#176b66");axes[1].set_title("Distances from one digit")
axes[2].bar(range(10),np.bincount(y),color="#d38b45");axes[2].set_title("Stratified class counts");plt.tight_layout();plt.show()

**Takeaway:** Digits vary in stroke style even within one class.

**Takeaway:** Raw high-dimensional distances span a broad, not self-interpreting scale.

**Takeaway:** Balanced labels make visual crowd size meaningful.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
Xs=StandardScaler().fit_transform(X);print("Scaled:",Xs.shape,"finite:",np.isfinite(Xs).all());pca=PCA(n_components=2,random_state=SEED).fit_transform(Xs)

# 6. Train

        Training turns the assumptions behind umap and t-sne versus pca into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. PCA preserves global linear variance.
2. t-SNE converts local similarities into probabilities and matches them in 2D.
3. UMAP builds a fuzzy nearest-neighbour graph and optimizes a low-dimensional graph.
4. Evaluate local neighbourhood preservation; do not infer global metric truth.

In [ ]:
um=umap.UMAP(n_neighbors=20,min_dist=.15,n_components=2,random_state=SEED,n_jobs=1).fit_transform(Xs);ts=TSNE(n_components=2,perplexity=30,init="pca",learning_rate="auto",max_iter=500,random_state=SEED).fit_transform(Xs)
emb={"PCA":pca,"UMAP":um,"t-SNE":ts};print("Embedding shapes:",{k:v.shape for k,v in emb.items()})

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
trust={k:trustworthiness(Xs,v,n_neighbors=10) for k,v in emb.items()};base=trustworthiness(Xs,np.random.default_rng(SEED).normal(size=(len(Xs),2)),n_neighbors=10)
fig,axes=plt.subplots(1,3,figsize=(14,3.7))
for ax,(name,z) in zip(axes,emb.items()): ax.scatter(z[:,0],z[:,1],c=y,cmap="tab10",s=7);ax.set_title(f"{name} · trust {trust[name]:.3f}");ax.set_xticks([]);ax.set_yticks([])
plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show();print("Random-layout trust baseline:",round(base,3))

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: Nonlinear embedding objectives prioritize neighbourhoods; random initialization can move, rotate, or resize clusters and alter distances between them. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
ts2=TSNE(n_components=2,perplexity=30,init="random",learning_rate="auto",max_iter=500,random_state=17).fit_transform(Xs)
def centroid_dist(z):
 c=np.vstack([z[y==k].mean(0) for k in range(10)]);return np.linalg.norm(c[:,None]-c[None,:],axis=2)[np.triu_indices(10,1)]
rho=np.corrcoef(centroid_dist(ts),centroid_dist(ts2))[0,1];local2=trustworthiness(Xs,ts2,n_neighbors=10)
print(f"t-SNE local trust run 1={trust['t-SNE']:.3f}, run 2={local2:.3f}; correlation of between-class distances={rho:.3f}")

**Use this when…** you need exploratory visualization of local high-dimensional neighbourhoods.

        **Don't use this when…** you need metric-preserving coordinates, reproducible global distances, causal clusters, or a visualization-free production feature without validation.

        ## Try this

        1. Sweep t-SNE perplexity and watch island layout change.
2. Sweep UMAP n_neighbors to trade local for broader structure.
3. Remove labels and decide what claims remain defensible.